In [1]:
import pandas as pd
import numpy as np
import sklearn
import xgboost as xgb
import sqlalchemy
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("sqlalchemy:", sqlalchemy.__version__)

pandas: 3.0.6
numpy: 2.4.6
scikit-learn: 1.9.1
xgboost: 3.2.0
sqlalchemy: 2.0.52


In [2]:
import sqlite3
conn = sqlite3.connect("../data/over25.db")
cursor = conn.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS test_table (id INTEGER PRIMARY KEY, note TEXT)")
cursor.execute("INSERT INTO test_table (note) VALUES ('setup test ok')")
conn.commit()
cursor.execute("SELECT * FROM test_table")
print(cursor.fetchall())
conn.close()

[(1, 'setup test ok'), (2, 'setup test ok')]


In [3]:
import pandas as pd
df = pd.read_csv("../data/raw/E0_2425.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(380, 120)
['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'BFH', 'BFD', 'BFA', 'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', '1XBH', '1XBD', '1XBA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'BFEH', 'BFED', 'BFEA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'BFE>2.5', 'BFE<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'BFEAHH', 'BFEAHA', 'B365CH', 'B365CD', 'B365CA', 'BWCH', 'BWCD', 'BWCA', 'BFCH', 'BFCD', 'BFCA', 'PSCH', 'PSCD', 'PSCA', 'WHCH', 'WHCD', 'WHCA', '1XBCH', '1XBCD', '1XBCA', 'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA', 'BFECH', 'BFECD', 'BFECA', 'B365C>2.5', 'B365C<2.5', 'PC>2.5', 'PC<2.5', 'MaxC>2.5', 'MaxC<2.5', 'AvgC>2.5', 'AvgC<2.5', 'BFEC>2.5', 'BFEC<2.5', 'AHCh', 'B365CAHH', 'B365CAHA', 'PCAHH

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,16/08/2024,20:00,Man United,Fulham,1,0,H,0,0,...,1.86,2.07,1.83,2.11,1.88,2.11,1.82,2.05,1.90,2.08
1,E0,17/08/2024,12:30,Ipswich,Liverpool,0,2,A,0,0,...,2.05,1.88,2.04,1.90,2.20,2.00,1.99,1.88,2.04,1.93
2,E0,17/08/2024,15:00,Arsenal,Wolves,2,0,H,1,0,...,2.02,1.91,2.00,1.90,2.05,1.93,1.99,1.87,2.02,1.96
3,E0,17/08/2024,15:00,Everton,Brighton,0,3,A,0,1,...,1.87,2.06,1.86,2.07,1.92,2.10,1.83,2.04,1.88,2.11
4,E0,17/08/2024,15:00,Newcastle,Southampton,1,0,H,1,0,...,1.87,2.06,1.88,2.06,1.89,2.10,1.82,2.05,1.89,2.10


In [4]:
df2 = pd.read_csv("../data/raw/E3_1718.csv")
print(df2.shape)
print(df2.columns.tolist())

(552, 65)
['Div', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 'VCH', 'VCD', 'VCA', 'Bb1X2', 'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA', 'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5', 'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA', 'PSCH', 'PSCD', 'PSCA']


In [5]:
import os

odds_summary = []
for fname in sorted(os.listdir("../data/raw")):
    if not fname.endswith(".csv") or fname.startswith("_"):
        continue
    path = os.path.join("../data/raw", fname)
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    has_bfe_match_odds = "BFEH" in cols
    has_bfe_ou25 = "BFE>2.5" in cols
    has_bb_ou25 = "BbAv>2.5" in cols
    has_avg_ou25 = "Avg>2.5" in cols
    odds_summary.append({
        "file": fname,
        "n_cols": len(cols),
        "BFE_match_odds": has_bfe_match_odds,
        "BFE_over_under_2.5": has_bfe_ou25,
        "Betbrain_over_under_2.5": has_bb_ou25,
        "Avg_over_under_2.5": has_avg_ou25,
    })

summary_df = pd.DataFrame(odds_summary)
summary_df

,file,n_cols,BFE_match_odds,BFE_over_under_2.5,Betbrain_over_under_2.5,Avg_over_under_2.5
0,E0_1718.csv,65,False,False,True,False
1,E0_1819.csv,62,False,False,True,False
2,E0_1920.csv,106,False,False,False,True
3,E0_2021.csv,106,False,False,False,True
4,E0_2122.csv,106,False,False,False,True
5,E0_2223.csv,106,False,False,False,True
6,E0_2324.csv,106,False,False,False,True
7,E0_2425.csv,120,True,True,False,True
8,E0_2526.csv,132,True,True,False,True
9,E1_1718.csv,65,False,False,True,False


In [6]:
# Compare one pre-COVID season against the two COVID-affected seasons
# and the first season back with fans, to see whether home advantage
# (and scoring) visibly shifted when stadiums were empty.
seasons_to_check = ["1819", "1920", "2021", "2122"]  # before, during, during/empty, after

for s in seasons_to_check:
    frames = []
    # Pull all four divisions for this season and stack them into one
    # table, since we want a single combined figure per season, not
    # per division.
    for div in ["E0", "E1", "E2", "E3"]:
        d = pd.read_csv(f"../data/raw/{div}_{s}.csv")
        frames.append(d)
    combined = pd.concat(frames)

    # Home win rate: fraction of matches where the full-time result (FTR) was "H"
    home_win_rate = (combined["FTR"] == "H").mean()

    # Average total goals per match: full-time home goals + full-time away goals
    avg_total_goals = (combined["FTHG"] + combined["FTAG"]).mean()

    print(f"{s}: home win rate = {home_win_rate:.3f}, avg total goals = {avg_total_goals:.3f}, n = {len(combined)}")

1819: home win rate = 0.440, avg total goals = 2.660, n = 2036
1920: home win rate = 0.435, avg total goals = 2.620, n = 1772
2021: home win rate = 0.401, avg total goals = 2.479, n = 2036
2122: home win rate = 0.436, avg total goals = 2.605, n = 2036


In [7]:
import pandas as pd
url = "https://www.football-data.co.uk/mmz4281/2627/E0.csv"
df_current = pd.read_csv(url)
print(df_current.shape)
df_current[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].tail()

(60, 114)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
55,10/10/2026,Man United,Tottenham,1,1,D
56,11/10/2026,Crystal Palace,Nott'm Forest,3,1,H
57,11/10/2026,Hull,Everton,2,2,D
58,11/10/2026,Liverpool,Man City,0,3,A
59,12/10/2026,Coventry,Newcastle,4,1,H


In [8]:
fixtures_url = "https://www.football-data.co.uk/fixtures.csv"
df_fixtures = pd.read_csv(fixtures_url)
print(df_fixtures.shape)
df_fixtures.head()

(169, 94)


,Div,Date,Time,HomeTeam,AwayTeam,Referee,B365H,B365D,B365A,BFDH,...,BFEC<2.5,AHCh,B365CAHH,B365CAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,D1,16/10/2026,19:30,Ein Frankfurt,FC Koln,NaN,1.85,4.20,3.60,1.83,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D1,17/10/2026,14:30,Elversberg,Augsburg,NaN,2.50,3.75,2.60,2.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D1,17/10/2026,14:30,Hamburg,Stuttgart,NaN,3.80,4.20,1.80,3.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D1,17/10/2026,14:30,Schalke 04,Mainz,NaN,3.10,3.75,2.25,2.88,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D1,17/10/2026,14:30,Union Berlin,Dortmund,NaN,5.50,4.33,1.57,5.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1]:
import pandas as pd
import os

all_teams = set()
for fname in sorted(os.listdir("../data/raw")):
    if fname.endswith(".csv") and not fname.startswith("_"):
        df = pd.read_csv(os.path.join("../data/raw", fname), usecols=["HomeTeam", "AwayTeam"])
        all_teams.update(df["HomeTeam"].dropna().unique())
        all_teams.update(df["AwayTeam"].dropna().unique())

print(len(all_teams))
print(sorted(all_teams))

103
['AFC Wimbledon', 'Accrington', 'Arsenal', 'Aston Villa', 'Barnet', 'Barnsley', 'Barrow', 'Birmingham', 'Blackburn', 'Blackpool', 'Bolton', 'Bournemouth', 'Bradford', 'Brentford', 'Brighton', 'Bristol City', 'Bristol Rvs', 'Bromley', 'Burnley', 'Burton', 'Bury', 'Cambridge', 'Cardiff', 'Carlisle', 'Charlton', 'Chelsea', 'Cheltenham', 'Chesterfield', 'Colchester', 'Coventry', 'Crawley Town', 'Crewe', 'Crystal Palace', 'Derby', 'Doncaster', 'Everton', 'Exeter', 'Fleetwood Town', 'Forest Green', 'Fulham', 'Gillingham', 'Grimsby', 'Harrogate', 'Hartlepool', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Leyton Orient', 'Lincoln', 'Liverpool', 'Luton', 'Macclesfield', 'Man City', 'Man United', 'Mansfield', 'Middlesbrough', 'Millwall', 'Milton Keynes Dons', 'Morecambe', 'Newcastle', 'Newport County', 'Northampton', 'Norwich', "Nott'm Forest", 'Notts County', 'Oldham', 'Oxford', 'Peterboro', 'Plymouth', 'Port Vale', 'Portsmouth', 'Preston', 'QPR', 'Reading', 'Rochdale', 'Rother

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/over25.db")
query = """
SELECT l.div_code, m.season, COUNT(*) as n_matches
FROM matches m
JOIN leagues l ON m.league_id = l.league_id
GROUP BY l.div_code, m.season
ORDER BY m.season, l.div_code
"""
counts = pd.read_sql(query, conn)
conn.close()
print(counts.to_string())

   div_code   season  n_matches
0        E0  2017/18        380
1        E1  2017/18        552
2        E2  2017/18        552
3        E3  2017/18        552
4        E0  2018/19        380
5        E1  2018/19        552
6        E2  2018/19        552
7        E3  2018/19        552
8        E0  2019/20        380
9        E1  2019/20        552
10       E2  2019/20        400
11       E3  2019/20        440
12       E0  2020/21        380
13       E1  2020/21        552
14       E2  2020/21        552
15       E3  2020/21        552
16       E0  2021/22        380
17       E1  2021/22        552
18       E2  2021/22        552
19       E3  2021/22        552
20       E0  2022/23        380
21       E1  2022/23        552
22       E2  2022/23        552
23       E3  2022/23        552
24       E0  2023/24        380
25       E1  2023/24        552
26       E2  2023/24        552
27       E3  2023/24        552
28       E0  2024/25        380
29       E1  2024/25        552
30      

## 2019/20 curtailed seasons (League One, League Two)

Row count sanity check (Phase 3) found League One at 400 matches and
League Two at 440, versus the expected 552 for a full season. This is
correct, not a data gap: the EFL voided the remainder of the 2019/20
season for League One and League Two due to COVID, deciding final
standings on points-per-game rather than completing the fixture list.
Premier League and Championship played their full seasons that year.
No missing data to chase here.

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/over25.db")

# Check for duplicate match_ids (shouldn't be possible given PRIMARY KEY, but confirm)
dupes = pd.read_sql("""
    SELECT match_id, COUNT(*) as n
    FROM matches
    GROUP BY match_id
    HAVING n > 1
""", conn)
print("Duplicate match_ids:", len(dupes))

# Check for genuine duplicate fixtures (same date/teams, different match_id -
# could happen if a team name was inconsistently cased/spaced)
near_dupes = pd.read_sql("""
    SELECT match_date, home_team_id, away_team_id, COUNT(*) as n
    FROM matches
    GROUP BY match_date, home_team_id, away_team_id
    HAVING n > 1
""", conn)
print("Same date/teams but different match_id:", len(near_dupes))

conn.close()

Duplicate match_ids: 0
Same date/teams but different match_id: 0


In [6]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/over25.db")

# Core match fields - these should basically never be missing
core_missing = pd.read_sql("""
    SELECT
        SUM(CASE WHEN home_goals IS NULL THEN 1 ELSE 0 END) as missing_home_goals,
        SUM(CASE WHEN away_goals IS NULL THEN 1 ELSE 0 END) as missing_away_goals,
        SUM(CASE WHEN result IS NULL THEN 1 ELSE 0 END) as missing_result,
        COUNT(*) as total_rows
    FROM matches
""", conn)
print("Core fields:")
print(core_missing.to_string(index=False))

# Match stats - known to be patchier in older seasons
stats_missing = pd.read_sql("""
    SELECT
        m.season,
        SUM(CASE WHEN s.home_shots IS NULL THEN 1 ELSE 0 END) as missing_shots,
        SUM(CASE WHEN s.home_corners IS NULL THEN 1 ELSE 0 END) as missing_corners,
        COUNT(*) as total_rows
    FROM matches m
    JOIN match_stats s ON m.match_id = s.match_id
    GROUP BY m.season
    ORDER BY m.season
""", conn)
print("\nMatch stats by season:")
print(stats_missing.to_string(index=False))

# Odds - we already know the coverage pattern, confirm it matches
odds_missing = pd.read_sql("""
    SELECT
        m.season,
        SUM(CASE WHEN o.avg_over_2_5 IS NULL THEN 1 ELSE 0 END) as missing_avg_odds,
        SUM(CASE WHEN o.bfe_over_2_5 IS NULL THEN 1 ELSE 0 END) as missing_bfe_odds,
        COUNT(*) as total_rows
    FROM matches m
    JOIN odds o ON m.match_id = o.match_id
    GROUP BY m.season
    ORDER BY m.season
""", conn)
print("\nOdds by season:")
print(odds_missing.to_string(index=False))

missing_bfe_detail = pd.read_sql("""
    SELECT l.div_code, m.season, COUNT(*) as n_missing
    FROM matches m
    JOIN leagues l ON m.league_id = l.league_id
    JOIN odds o ON m.match_id = o.match_id
    WHERE o.bfe_over_2_5 IS NULL
      AND m.season IN ('2024/25', '2025/26')
    GROUP BY l.div_code, m.season
    ORDER BY m.season, l.div_code
""", conn)
print(missing_bfe_detail.to_string(index=False))

conn.close()

Core fields:
 missing_home_goals  missing_away_goals  missing_result  total_rows
                  0                   0               0       18060

Match stats by season:
 season  missing_shots  missing_corners  total_rows
2017/18              0                0        2036
2018/19              1                1        2036
2019/20              0                0        1772
2020/21              0                0        2036
2021/22              0                0        2036
2022/23              0                0        2036
2023/24              0                0        2036
2024/25              0                0        2036
2025/26              0                0        2036

Odds by season:
 season  missing_avg_odds  missing_bfe_odds  total_rows
2017/18              2036              2036        2036
2018/19              2036              2036        2036
2019/20                 5              1772        1772
2020/21                 0              2036        2036
2021/22   

## Missing Betfair Exchange odds within 2024/25-2025/26

Even within the two seasons that generally have BFE Over/Under 2.5 data,
a small number of matches are missing it: 24 in 2024/25, 118 in 2025/26
(spread across all four divisions, skewing slightly toward League Two
and toward 2025/26 since it's still in progress). Likely cause: Betfair
Exchange is a liquidity-driven market, not a fixed bookmaker price list -
some fixtures (especially lower-liquidity lower-league ones) simply
don't get a settled price captured in the source. Affects Phase 10
(value/betting backtest) only, not the core goals model. These matches
will be excluded from EV calculations rather than filled with any
assumed value.